# 03a | Financial-domain training from completed pretraining

130U-FinLLM | 130U Financial Language Model

Read and verify the Drive checkpoint with at least 1,220 expansion updates, its data and the original 8K tokenizer,
and freeze the parent checkpoint before training. By default reuse `expansion_v1/domain_mix`,
whose 1,215,545 tokens contain approximately 80% financial and 20% general content.

This is a new `domain_v1` stage: initially import only parent weights and create a new optimizer and learning-rate schedule,
starting stage updates at zero. Existing checkpoints for this stage restore complete state strictly, without reinitialization.
Select Runtime > Change runtime type > T4 GPU and run in order from the first cell.

By default verify update 1, then train to 74 cumulative stage updates, nominally about 1,212,416 token positions.
This is a bounded experiment on existing corpora; it neither completes the 25M domain budget nor measures asset-stance accuracy.
Do not repeat 01b, 02c or 02d, or start another training process concurrently.

In [ ]:
import os, sys, json, shutil, subprocess, importlib
from pathlib import Path

REPOSITORY = "https://github.com/130U/130U-FinLLM.git"
REF = "main"  # Resolve and display the commit first; use the full 40-character SHA for reproduction.
PROJECT = Path("/content/130U-FinLLM")

def command(*args, cwd=None):
    # Explicitly relay subprocess output into the notebook's captured stdout.
    with subprocess.Popen([str(x) for x in args], cwd=cwd, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, encoding="utf-8",
                          errors="replace", bufsize=1) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
        if process.wait():
            raise subprocess.CalledProcessError(process.returncode, process.args)

if len(REF) == 40 and all(c in "0123456789abcdefABCDEF" for c in REF):
    REVISION = REF.lower()
else:
    refs = subprocess.check_output(["git", "ls-remote", REPOSITORY, REF], text=True).strip().splitlines()
    if len(refs) != 1:
        raise RuntimeError("REF does not resolve uniquely; enter the full commit SHA.")
    REVISION = refs[0].split()[0]

if not PROJECT.exists():
    command("git", "clone", "--no-checkout", REPOSITORY, PROJECT)
else:
    remote = subprocess.check_output(["git", "remote", "get-url", "origin"], cwd=PROJECT, text=True).strip()
    if remote.rstrip("/").removesuffix(".git") != REPOSITORY.removesuffix(".git"):
        raise RuntimeError("/content/130U-FinLLM belongs to another repository; choose a new directory.")
command("git", "fetch", "--depth", "1", "origin", REVISION, cwd=PROJECT)
command("git", "checkout", "--detach", REVISION, cwd=PROJECT)
os.chdir(PROJECT)
print("Pinned code revision:", REVISION)

# Keep the bundled Colab CUDA / PyTorch; install only the other project dependencies.
import torch
from importlib import metadata
from packaging.requirements import Requirement

def tokenizer_install_requirements():
    # Colab can include Transformers even though this project does not use it.
    # Pass its active tokenizer bounds to pip so an old runtime can retain 0.22,
    # while a current runtime (or an already broken one) can use 0.23.
    requirements = ["tokenizers>=0.21,<0.24"]
    try:
        existing = metadata.requires("transformers") or []
    except metadata.PackageNotFoundError:
        existing = []
    for value in existing:
        requirement = Requirement(value)
        if requirement.name.lower().replace("_", "-") == "tokenizers":
            if requirement.marker is None or requirement.marker.evaluate({"extra": ""}):
                requirements.append(str(requirement))
    return list(dict.fromkeys(requirements))

def check_tokenizer_compatibility(requirements):
    installed = metadata.version("tokenizers")
    incompatible = [value for value in requirements
                    if installed not in Requirement(value).specifier]
    if incompatible:
        raise RuntimeError(f"tokenizers {installed} is incompatible with the existing environment:{incompatible}")
    print("tokenizers compatibility check passed:", installed)

def check_project_dependencies():
    # Read the refreshed installed metadata, not the previous editable checkout.
    problems = []
    for value in metadata.requires("finllm-130u") or []:
        requirement = Requirement(value)
        if requirement.marker is not None and not requirement.marker.evaluate({"extra": ""}):
            continue
        try:
            installed = metadata.version(requirement.name)
        except metadata.PackageNotFoundError:
            problems.append(f"Missing {requirement.name}")
            continue
        if installed not in requirement.specifier:
            problems.append(f"{requirement.name}=={installed} does not satisfy {requirement}")
    if problems:
        raise RuntimeError("Project dependency check failed:" + "; ".join(problems))
    print("Installed project dependencies passed validation.")

def activate_project_in_kernel():
    # pip installs the editable .pth for NEW Python processes. A notebook kernel
    # that was already running does not process that .pth again automatically.
    source = (PROJECT / "src").resolve()
    expected = source / "finllm"
    if not (expected / "__init__.py").is_file():
        raise RuntimeError(f"Project Python source is missing:{expected}")
    sys.path[:] = [str(source)] + [entry for entry in sys.path if entry != str(source)]
    # Git may have changed code in the same directory since an earlier cell run.
    # Refresh only our modules; never reload Torch or its live CUDA state.
    for name in tuple(sys.modules):
        if name == "finllm" or name.startswith("finllm."):
            del sys.modules[name]
    importlib.invalidate_caches()
    package = importlib.import_module("finllm")
    runtime = importlib.import_module("finllm.runtime")
    for module, filename in ((package, "__init__.py"), (runtime, "runtime.py")):
        if Path(module.__file__).resolve() != expected / filename:
            raise RuntimeError(f"The notebook imported the wrong project location:{module.__file__}")
    if not callable(runtime.restore_seed_checkpoint):
        raise RuntimeError("The checkpoint restore interface is unavailable in this notebook.")
    print("Current notebook kernel import check passed:", runtime.__file__)

def check_fresh_process_import():
    # Verify the installed editable package separately, without injecting src
    # into PYTHONPATH. Both the current kernel and future CLI processes must work.
    probe = """import json, sys
from pathlib import Path
import finllm
import finllm.runtime as runtime
expected = Path(sys.argv[1]).resolve() / 'finllm'
for module, filename in ((finllm, '__init__.py'), (runtime, 'runtime.py')):
    if Path(module.__file__).resolve() != expected / filename:
        raise RuntimeError('Fresh process imported the wrong project: ' + str(module.__file__))
if not callable(runtime.restore_seed_checkpoint):
    raise RuntimeError('Fresh process checkpoint restore is unavailable')
print(json.dumps({'finllm': str(Path(finllm.__file__).resolve()), 'runtime': str(Path(runtime.__file__).resolve())}))
"""
    result = subprocess.check_output([sys.executable, "-c", probe, str(PROJECT / "src")],
                                     cwd=PROJECT, text=True)
    print("Fresh Python process import check passed:", result.strip())

if tuple(int(v) for v in torch.__version__.split("+")[0].split(".")[:2]) < (2, 5):
    raise RuntimeError("PyTorch >= 2.5 is required; select a newer Colab runtime.")
tokenizer_requirements = tokenizer_install_requirements()
print("Active tokenizers constraints:", tokenizer_requirements)
torch_version_before = metadata.version("torch")
torch_file_before = Path(torch.__file__).resolve()
# One resolver transaction replaces the previous finllm metadata AND its
# dependencies. Upgrading tokenizers first leaves old finllm bounds active and
# produces a misleading conflict warning even when the eventual install works.
# The exact installed Torch build is constrained, so pip keeps Colab's CUDA build.
command(sys.executable, "-m", "pip", "install", "-q", "-e", ".",
        "torch==" + torch_version_before, *tokenizer_requirements)
importlib.invalidate_caches()
if metadata.version("torch") != torch_version_before or Path(torch.__file__).resolve() != torch_file_before:
    raise RuntimeError("The PyTorch environment changed unexpectedly; preserve the output and stop training.")
check_tokenizer_compatibility(tokenizer_requirements)
check_project_dependencies()
activate_project_in_kernel()
check_fresh_process_import()
print("Notebook setup complete; PyTorch remains:", torch_version_before)

def finllm(*args):
    command(sys.executable, "-u", "-m", "finllm", *args, cwd=PROJECT)

def require(*paths):
    missing = [str(p) for p in paths if not Path(p).is_file()]
    if missing:
        raise FileNotFoundError("Stage inputs are missing; prepare them or restore them from Drive first:\n" + "\n".join(missing))

finllm("environment")
disk = shutil.disk_usage("/content")
print(f"Free disk space in /content:{disk.free / 2**30:.2f} GiB")
Path("runtime_revision.txt").write_text(REVISION + "\n")

In [ ]:
MOUNT_DRIVE = True
if not MOUNT_DRIVE:
    raise RuntimeError("This stage requires existing seed inputs and versioned backups; mount Drive first.")
from google.colab import drive
drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/130U-FinLLM")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print("Drive project directory:", DRIVE_ROOT)

In [ ]:
DOMAIN_INPUTS_VERIFIED = False
DOMAIN_RETENTION_BEFORE_SUCCEEDED = False
DOMAIN_PROBE_SUCCEEDED = False
DOMAIN_TRAIN_SUCCEEDED = False
DOMAIN_RETENTION_AFTER_SUCCEEDED = False
DOMAIN_BACKED_UP = False
SEED_VERSION = "seed_v1"
EXPANSION_VERSION = "expansion_v1"
DOMAIN_VERSION = "domain_v1"
MIN_PARENT_UPDATES = 1220
PROBE_UPDATES = 1
TARGET_UPDATES = 74
if not torch.cuda.is_available():
    raise RuntimeError("Select a T4 GPU runtime; domain training requires a GPU.")
from finllm.domain_runtime import (
    prepare_domain_training, run_domain_segment,
    retention_domain_check, backup_domain_training)
from finllm.checkpoint import load_checkpoint, sha256_file
domain = prepare_domain_training(
    PROJECT, DRIVE_ROOT, DOMAIN_VERSION, EXPANSION_VERSION, SEED_VERSION,
    min_parent_updates=MIN_PARENT_UPDATES, probe_updates=PROBE_UPDATES,
    target_updates=TARGET_UPDATES)
# Read the verified frozen parent checkpoint directly; an old screenshot is not a current Drive read.
parent_checkpoint = Path(domain["config"]["init_checkpoint"])
parent_payload = load_checkpoint(parent_checkpoint, allow_previous=False)
parent_audit = {
    "frozen_checkpoint": str(parent_checkpoint),
    "checkpoint_sha256": sha256_file(parent_checkpoint),
    "stage": parent_payload["config"]["stage"],
    "updates": parent_payload["progress"]["updates"],
    "tokens": parent_payload["progress"]["tokens"],
    "tokenizer_sha256": parent_payload["tokenizer_hash"],
    "data_sha256": parent_payload["provenance"]["data_sha256"],
    "validation_sha256": parent_payload["provenance"]["validation_sha256"],
}
if parent_audit["stage"] != "pretrain" or parent_audit["updates"] < MIN_PARENT_UPDATES:
    raise ValueError("The parent checkpoint is not the required completed pretraining artifact; preserve existing files and stop.")
print("Currently verified Drive parent checkpoint:", json.dumps(parent_audit, indent=2, ensure_ascii=False))
print("Current cumulative domain updates:", domain["restored_updates"])
print("Stage configuration:", json.dumps(domain["config"], indent=2, ensure_ascii=False))
print("GPU:", torch.cuda.get_device_name(0))
DOMAIN_INPUTS_VERIFIED = True

The previous cell only validates inputs, freezes parent artifacts and prepares configuration; optimizer updates have not started.
Keep the actual parent SHA256, update count and tokens as evidence of this cloud read.
The diagnostics below use frozen validation texts. Diagnostic and training loss measure next-token prediction,
not bullish/bearish scores. Validation failure prevents subsequent training and complete backups.

In [ ]:
DOMAIN_RETENTION_BEFORE_SUCCEEDED = False
DOMAIN_PROBE_SUCCEEDED = False
DOMAIN_TRAIN_SUCCEEDED = False
DOMAIN_RETENTION_AFTER_SUCCEEDED = False
DOMAIN_BACKED_UP = False
if not globals().get("DOMAIN_INPUTS_VERIFIED", False):
    raise RuntimeError("Validate current Drive inputs and freeze the parent checkpoint first.")
domain_retention_before = retention_domain_check(domain, "before")
print(json.dumps(domain_retention_before, indent=2, ensure_ascii=False))
DOMAIN_RETENTION_BEFORE_SUCCEEDED = True

In [ ]:
DOMAIN_PROBE_SUCCEEDED = False
DOMAIN_TRAIN_SUCCEEDED = False
DOMAIN_RETENTION_AFTER_SUCCEEDED = False
DOMAIN_BACKED_UP = False
if not globals().get("DOMAIN_INPUTS_VERIFIED", False) or not globals().get("DOMAIN_RETENTION_BEFORE_SUCCEEDED", False):
    raise RuntimeError("Validate the parent checkpoint and complete pretraining diagnostics first.")
domain_probe = run_domain_segment(domain, "probe", finllm)
print(json.dumps(domain_probe, indent=2, ensure_ascii=False))
DOMAIN_PROBE_SUCCEEDED = True

After one-step verification, continue to 74 cumulative stage updates. Record parent pretraining and domain tokens separately,
with a separate total along the model lineage. Preserve original seed and expansion artifacts, the 1,220-update parent checkpoint and its reports.
Handle initial domain weight transfer separately from strict recovery. Incompatible data, tokenizer, configuration or stage runtime stops execution.
After disconnection, check prior execution status before recovering from the first cell; preserve existing checkpoints and reports.

In [ ]:
DOMAIN_TRAIN_SUCCEEDED = False
DOMAIN_RETENTION_AFTER_SUCCEEDED = False
DOMAIN_BACKED_UP = False
if not globals().get("DOMAIN_PROBE_SUCCEEDED", False):
    raise RuntimeError("Complete the domain-stage one-step check successfully first.")
domain_result = run_domain_segment(domain, "continue", finllm)
print(json.dumps(domain_result, indent=2, ensure_ascii=False))
DOMAIN_TRAIN_SUCCEEDED = True

In [ ]:
DOMAIN_RETENTION_AFTER_SUCCEEDED = False
DOMAIN_BACKED_UP = False
if not globals().get("DOMAIN_TRAIN_SUCCEEDED", False):
    raise RuntimeError("Stage training has not completed successfully; keep existing backups.")
domain_retention_after = retention_domain_check(domain, "after")
print(json.dumps(domain_retention_after, indent=2, ensure_ascii=False))
DOMAIN_RETENTION_AFTER_SUCCEEDED = True

In [ ]:
DOMAIN_BACKED_UP = False
if not all(globals().get(flag, False) for flag in (
    "DOMAIN_INPUTS_VERIFIED", "DOMAIN_RETENTION_BEFORE_SUCCEEDED",
    "DOMAIN_PROBE_SUCCEEDED", "DOMAIN_TRAIN_SUCCEEDED", "DOMAIN_RETENTION_AFTER_SUCCEEDED")):
    raise RuntimeError("Input checks, diagnostics and training have not all succeeded; do not report a complete stage backup.")
domain_backup = backup_domain_training(domain)
print(json.dumps(domain_backup, indent=2, ensure_ascii=False))
shutil.copy2(PROJECT / "runtime_revision.txt", domain["paths"]["drive"] / "runtime_revision.txt")
DOMAIN_BACKED_UP = True
print("Financial-domain training and verified backup complete:", domain["paths"]["drive"] / "run")
print("Domain-stage reports and diagnostics:", domain["paths"]["drive"])
print("Complete parent checkpoint frozen:", parent_checkpoint)

Completion criteria: actual parent identity has been read; domain one-step and cumulative 74-update reports are verifiable;
before/after diagnostics are retained; the final backup reports `verified: true`.
Rerunning a completed stage reuses original checkpoints and measurements without timing it again or overwriting reports.
This notebook completes a bounded financial-corpus experiment; the original 100M/25M compute budgets remain separate.
SFT, DPO, calibration and asset-stance evaluation require target-level labels and review records consistent with the project task definition.